# Reusable Mention Engine and Notebook Integration

**Status:** Proposed design for review  
**Date:** 2026-08-27  
**Design epic:** `bd-2n81`  
**Authoritative artifact:** this notebook

## Decision summary

Extract a dependency-light `spur-mentions` engine and keep product/runtime policy in adapters:

- `spur-mentions`: neutral candidates, source snapshots, root-scoped caching, configurable filesystem traversal, Nucleo ranking, bounded top-K selection, and optional code-graph discovery/hydration/expansion.
- `spur-tui`: Ratatui presentation, `MentionQuerySource`, background scheduling, `Rc<RefCell<_>>`, worker/issue/datasource adapters, worker token composition, picker sections, protected ranges, and ACP assembly.
- `spur-notebook`: root selection, a thin completion adapter, persistent `Arc<Mutex<MentionEngine>>` state, mention-range validation, and ACP prompt composition.

The design was selected section-by-section with Z3 Optimize. Native `ns_mermaid` cells below are the binding architecture, protocol, and lifecycle contracts.

## Scope, evidence, goals, and non-goals

### Current evidence

- `crates/spur-tui/src/mentions/registry.rs` is approximately 4,000 lines and combines source registration, cache lifetime, code-graph resolution, worker composition, model probing, ranking, and picker layout.
- Typed queries score all ordinary entries and code candidates, fully sort the match vector, and only then truncate.
- `MentionEntry` contains ACP worker fields, issue preview payloads, section-header state, and InputBar suffix state.
- The notebook helper at `/Volumes/Projects/Projects/spur-notebook/src/commands/chat/helpers.inc.rs:868` rescans the canonical workspace for each request.
- Notebook range validation and ACP block composition live independently at `sidebar_chat/manager.rs:2974` and must remain there.

### Goals

1. One reusable completion engine for TUI and notebook consumers.
2. No Ratatui, ACP prompt-composition, `Rc<RefCell<_>>`, or notebook range dependency in the shared crate.
3. Behavior profiles make all traversal/ranking differences explicit.
4. Warm queries reuse root-scoped snapshots.
5. Bounded ranking avoids a full `O(N log N)` sort while preserving result order.
6. Code-mention discovery, lazy hydration, and expansion are reusable behind an optional feature.
7. Migration has parity gates and rollback points.

### Non-goals

- Replacing notebook root-selection policy.
- Moving mention-range validation or ACP composition into the shared crate.
- Making worker/issue/datasource models universal on the first extraction.
- Building a sublinear search index in this change.
- Changing ACP envelopes or protected-range behavior.
- Refactoring unrelated TUI input/rendering code.

## 1. Ownership architecture

**Z3 Optimize decision:** `split_engine` — solve `sol_4dcc1f84ce6f4b24`.

The optimized model satisfied mode independence (12), shared ranking/code reuse (10), and behavior preservation (9), while accepting the smaller-diff preference as the only violated soft constraint (4).

### Binding ownership rules

- Dependency direction is consumer → shared engine → optional external libraries.
- `spur-mentions` must never depend on `spur-tui` or `spur-notebook`.
- TUI and notebook controllers own lifecycle/state handles; the engine owns completion mechanics.
- ACP and protected-range composition remain at each frontend boundary.
- Code-graph functionality is optional so filesystem-only consumers do not inherit `spur-graph` types.
- A TUI `MentionRegistry` façade preserves the existing caller surface while delegating neutral work to `MentionEngine`.

The following native architecture cell is authoritative for service placement and allowed cross-zone edges. Same-zone edges are allowed; the only cross-zone directions are TUI → shared, notebook → shared, and shared → declared dependencies.

In [ ]:
architecture-beta
    group tui(cloud)[spur-tui]
    group shared(cloud)[spur-mentions]
    group notebook(cloud)[spur-notebook]
    group external(cloud)[Dependencies]

    service query_source(server)[MentionQuerySource] in tui
    service tui_registry(server)[TUI MentionRegistry facade] in tui
    service tui_adapters(server)[Worker Issue Datasource adapters] in tui
    service tui_submit(server)[ProtectedRange and ACP assembly] in tui

    service engine(server)[MentionEngine] in shared
    service entry(server)[Neutral MentionEntry] in shared
    service file_source(disk)[FileMentionSource profiles] in shared
    service cache(database)[Root scoped snapshot cache] in shared
    service ranker(server)[Nucleo top K ranker] in shared
    service code_mentions(server)[Code source hydration expansion] in shared

    service notebook_command(server)[mention_complete_in_workspace adapter] in notebook
    service notebook_state(database)[Arc Mutex MentionEngine state] in notebook
    service sidebar_manager(server)[Range validation and ACP composition] in notebook

    service ignore_dep(server)[ignore] in external
    service nucleo_dep(server)[nucleo matcher] in external
    service graph_dep(server)[spur-graph optional feature] in external

    %% @ns-zone-allow tui shared
    %% @ns-zone-allow notebook shared
    %% @ns-zone-allow shared external

    query_source:R --> L:tui_registry
    tui_adapters:R --> L:tui_registry
    tui_registry:R --> L:engine
    tui_submit:T --> B:tui_registry

    notebook_command:R --> L:notebook_state
    notebook_state:T --> B:engine

    engine:R --> L:entry
    engine:R --> L:file_source
    engine:R --> L:cache
    engine:R --> L:ranker
    engine:R --> L:code_mentions

    file_source:R --> L:ignore_dep
    ranker:R --> L:nucleo_dep
    code_mentions:R --> L:graph_dep

## 2. Public API and data model

**Z3 Optimize decision:** `neutral_core_sidecar` — solve `sol_080f3c2d00ed44f3`.

This satisfies dependency neutrality (12), typed extension (9), and object-safe source composition (7), at the cost of some adapter mapping (4).

### Shared core

`MentionEntry` is reduced to mode-neutral ranking and insertion data:

| Field | Contract |
|---|---|
| `id: MentionId` | Stable within one source snapshot; suitable as a sidecar key |
| `kind: MentionKind` | File, directory, code file, code symbol, or caller-defined neutral category |
| `uri: String` | Canonical resource identity |
| `display: String` | Primary picker label |
| `secondary: Option<String>` | Optional neutral detail |
| `search_text: Option<String>` | Explicit ranking haystack |
| `insert_text: Option<String>` | Text inserted by the consumer; no range semantics |

`section_header`, worker agent/model/effort, `AgentKind`, worker CLI identity, issue previews, tags, and unconsumed suffixes do not belong in this type.

### Source and engine contracts

- `MentionSource: Send` exposes only `key()` and `build(root, context) -> SourceSnapshot`.
- `SourceSnapshot` contains neutral entries and a monotonically increasing generation.
- Code capabilities are isolated in the optional `code` module: `CodeMentionSource`, `CodePayloadStore`, hydration, validation, and expansion.
- `MentionEngine::query(root, query, options)` returns `QueryResult { entries, diagnostics }`.
- Errors are typed as root, traversal, source-build, ranking, and code-hydration failures.
- Source failure policy is explicit: required sources fail the query; optional sources contribute diagnostics and allow partial results.

### TUI sidecars

`TuiMentionMetadata`, keyed by `MentionId`, retains worker composition state, issue previews, tags, code preview details, and any InputBar-specific information. Datasource prompt hints stay in their adapter-owned store.

## 3. Filesystem and ranking compatibility profiles

**Z3 Optimize decision:** `compatibility_profiles` — solve `sol_d19463fd3c224a70`.

The selected model preserves both clients (12), prevents silent behavior changes (10), and retains one shared source implementation (8). It deliberately accepts a slightly larger public options surface (4).

### Required profiles

| Policy | `TuiProfile` | `NotebookCompatibilityProfile` |
|---|---|---|
| Files | Include | Include |
| Directories | Include | Exclude |
| Hidden entries | Exclude | Include |
| Git/global/parent ignore rules | Respect | Do not apply |
| Symlink traversal | Do not follow | Do not follow |
| URI construction | Standards-based file URL | Standards-based file URL |
| Match mode | Nucleo smart fuzzy | Case-insensitive substring initially |
| Tie ordering | Existing TUI comparator | Prefix-first, lowercase lexical |
| Result policy | Caller limit plus TUI section policy | 100 |

Profile identity participates in the cache key. A snapshot produced with one traversal profile cannot be reused under another profile.

The shared implementation must use `url::Url::from_file_path` or an equivalent validated encoder. If changing TUI's existing manual URI formatting affects spaces or non-ASCII paths, that change requires an explicit compatibility test.

A later product decision may switch the notebook to Nucleo ranking, but it is not bundled into the extraction. First integration must pass the existing notebook completion tests unchanged.

## 4. Cache ownership and concurrency

**Z3 Optimize decision:** `injected_arc_engine` — solve `sol_b1181d8ef6b54c4d`.

It satisfies explicit lifecycle (12), `spawn_blocking` safety (10), and test isolation (9). It rejects a hidden process-global cache despite its smaller wiring cost (4).

### Runtime model

- The shared engine is `Send`; frontends choose their local ownership wrapper.
- TUI retains `Rc<RefCell<TuiMentionRegistry>>` because it is UI-thread state. Background work receives immutable query work or `Send` source/cache handles, never the `Rc`.
- Notebook application state owns `Arc<Mutex<MentionEngine>>` and passes it into the blocking completion task.
- Constructing an engine inside every completion call is forbidden because it defeats warm caching.
- Process-global caches are forbidden: they obscure lifetime, complicate tests, and make workspace eviction implicit.

### Cache identity and invalidation

`CacheKey = canonical_root + source_key + profile_fingerprint + source_token`.

- Filesystem snapshots use TTL plus explicit root invalidation.
- Code snapshots include graph pointer/artifact identity.
- Snapshot publication is atomic: readers see either the previous complete generation or the new complete generation.
- Failed builds never replace a valid previous snapshot.
- Cache capacity and eviction policy are configuration, not UI constants.
- The initial compatibility TTL is 600 seconds, matching the current TUI registry; benchmarks may justify changing it in a later decision.

The following formal lifecycle proves non-negative, monotonically ordered snapshot generations across start, publish, refresh, invalidate, and rebuild transitions.

In [ ]:
stateDiagram-v2
    [*] --> Cold
    Cold --> Building: start
    Building --> Ready: publish
    Ready --> Building: refresh
    Ready --> Stale: invalidate
    Stale --> Building: rebuild

    note right of Cold
      @spec MENTION-CACHE-LIFECYCLE
      @type CacheEvent = enum[start,publish,refresh,invalidate,rebuild]
      @input event: CacheEvent
      @state-var generation: Int
      @state-var published_generation: Int
      @state-var building: Bool
      @requires PRE: generation = 0 and published_generation = 0 and building = false
      @state Cold
      @invariant MONOTONIC: generation >= 0 and published_generation >= 0 and published_generation <= generation
    end note

    note right of Building
      @state Building
      @transition START
      @from Cold
      @to Building
      @event event = start
      @guard building = false
      @update generation' = generation + 1
      @update published_generation' = published_generation
      @update building' = true
      @verify INIT: prove initiate MONOTONIC
      @verify PRESERVE_START: prove preserve MONOTONIC on START
    end note

    note right of Ready
      @state Ready
      @transition PUBLISH
      @from Building
      @to Ready
      @event event = publish
      @guard building = true
      @update generation' = generation
      @update published_generation' = generation
      @update building' = false
      @verify PRESERVE_PUBLISH: prove preserve MONOTONIC on PUBLISH
    end note

    note right of Building

      @transition REFRESH
      @from Ready
      @to Building
      @event event = refresh
      @guard building = false
      @update generation' = generation + 1
      @update published_generation' = published_generation
      @update building' = true
      @verify PRESERVE_REFRESH: prove preserve MONOTONIC on REFRESH
    end note

    note right of Stale
      @state Stale
      @transition INVALIDATE
      @from Ready
      @to Stale
      @event event = invalidate
      @guard building = false
      @update generation' = generation + 1
      @update published_generation' = published_generation
      @update building' = false
      @verify PRESERVE_INVALIDATE: prove preserve MONOTONIC on INVALIDATE
    end note

    note right of Building

      @transition REBUILD
      @from Stale
      @to Building
      @event event = rebuild
      @guard building = false
      @update generation' = generation
      @update published_generation' = published_generation
      @update building' = true
      @verify PRESERVE_REBUILD: prove preserve MONOTONIC on REBUILD
    end note

## 5. Ranking and completion complexity

**Z3 Optimize decision:** `select_nth_then_sort_k` — solve `sol_d797f83261e24555`.

The model preserves the exact comparator (12), improves on the full sort (11), minimizes migration risk (9), and avoids a new index (7). It does not claim sublinear completion (violated preference 3).

### Algorithm

For a typed query with `N` candidates, `M` matches, and output limit `K`:

1. Score every candidate and collect `M` ranked references: `O(N)`.
2. Determine the global maximum rank required by the current score-bucket comparator.
3. If `M > K`, partition with `select_nth_unstable_by(K, comparator)`: expected `O(M)`.
4. Sort only the selected prefix: `O(K log K)`.
5. Hydrate code payloads only for selected code rows.

Expected ranking complexity becomes `O(N + K log K)`; memory remains `O(M)`. The full `O(N)` scoring pass remains. Sublinear queries require a separate indexed-search design.

### Exactness gate

A property test must compare `rank_top_k(entries, query, K)` against the first `K` rows of the current full-sort implementation across:

- random scores and stable tie keys,
- all mention kinds,
- composed-worker substitution,
- code candidates mixed with ordinary entries,
- `K = 0`, `K = 1`, `K >= M`,
- equal-rank and equal-display cases.

Empty-query section selection remains TUI policy. It may reuse the shared bounded-selection primitive but must not move section headers or caps into the engine.

## 6. Notebook integration and protocol

**Z3 Optimize decision:** `adapter_exact_rev` — solve `sol_e72408ccc1a94f60`.

The selected seam preserves manager ownership (12), isolates dependency versions (10), and uses the shared engine (9), while accepting a small mapping layer (4).

### Completion boundary

Only the completion implementation changes semantically:

1. Existing notebook scope/root selection chooses the canonical root.
2. The command obtains the persistent engine handle from application state.
3. A blocking task locks the engine, applies `NotebookCompatibilityProfile`, and queries with limit 100.
4. Neutral entries are mapped into `ChatMentionCompletion`.
5. No prompt text, mention byte ranges, attachments, or ACP blocks enter `spur-mentions`.

`validated_mention_ranges` and `compose_prompt_blocks` in `sidebar_chat/manager.rs` remain unchanged and continue to own UTF-8 boundary checks, overlap checks, resource-link construction, attachment ordering, and ACP framing.

### Dependency seam

- Add `spur-mentions` as an exact Git revision.
- Enable only filesystem/ranking features initially; leave `code-graph` disabled in the notebook.
- If the notebook later consumes shared code payload types, align `spur-mentions` and `spur-graph` to one immutable revision.
- The shared crate must not expose `spur-acp` types, preventing duplicate-version type identity problems.
- Confirm existing GPL dependency/distribution policy before release; the notebook already consumes SPUR crates, but the new crate must follow the same reviewed policy.

The sequence cell formalizes that completion terminates at neutral adapter rows, while range validation and ACP composition remain a separate manager-owned submission step.

In [ ]:
sequenceDiagram
    participant UI as Notebook UI
    participant Command as Chat command
    participant Engine as MentionEngine
    participant Files as FileMentionSource
    participant Manager as Sidebar manager

    Note over UI,Manager: @spec NOTEBOOK-MENTION-INTEGRATION
    UI->>Command: request_completion
    Note over UI,Command: @message REQUEST_COMPLETION<br/>@from UI<br/>@to Command<br/>@event request_completion<br/>@order 1<br/>@when true<br/>@ensures REQUEST_ACCEPTED: true
    Command->>Engine: query_engine
    Note over Command,Engine: @message QUERY_ENGINE<br/>@from Command<br/>@to Engine<br/>@event query_engine<br/>@order 2<br/>@when true<br/>@ensures ROOT_SELECTED_OUTSIDE_ENGINE: true
    Engine->>Files: ensure_snapshot
    Note over Engine,Files: @message ENSURE_SNAPSHOT<br/>@from Engine<br/>@to Files<br/>@event ensure_snapshot<br/>@order 3<br/>@when true<br/>@ensures PROFILE_EXPLICIT: true
    Files-->>Engine: return_snapshot
    Note over Files,Engine: @message RETURN_SNAPSHOT<br/>@from Files<br/>@to Engine<br/>@event return_snapshot<br/>@order 4<br/>@when true<br/>@ensures SNAPSHOT_NEUTRAL: true
    Engine-->>Command: return_candidates
    Note over Engine,Command: @message RETURN_CANDIDATES<br/>@from Engine<br/>@to Command<br/>@event return_candidates<br/>@order 5<br/>@when true<br/>@ensures LIMIT_APPLIED: true
    Command-->>UI: map_completions
    Note over Command,UI: @message MAP_COMPLETIONS<br/>@from Command<br/>@to UI<br/>@event map_completions<br/>@order 6<br/>@when true<br/>@ensures NO_RANGE_COMPOSITION: true
    UI->>Manager: submit_prompt
    Note over UI,Manager: @message SUBMIT_PROMPT<br/>@from UI<br/>@to Manager<br/>@event submit_prompt<br/>@order 7<br/>@when true<br/>@ensures RANGE_OWNER_IS_MANAGER: true
    Manager-->>UI: return_blocks
    Note over Manager,UI: @message RETURN_BLOCKS<br/>@from Manager<br/>@to UI<br/>@event return_blocks<br/>@order 8<br/>@when true<br/>@ensures ACP_OWNER_IS_MANAGER: true
    Note over UI,Manager: @verify PROTOCOL: prove sequence_protocol

## Error handling and observability

Errors are attributable to one boundary and must not be flattened into an empty picker:

| Failure | Owner | Result |
|---|---|---|
| Invalid or missing canonical root | Notebook/TUI caller | Existing root error/hint |
| Required file-source build fails | Engine | Typed query error |
| Optional code graph missing | Code adapter | Partial results plus diagnostic hint |
| Code hydration fails for a selected row | Code adapter | Row-level warning or omission according to current TUI policy |
| Cache mutex poisoned/closed | Frontend state owner | Recoverable command error; no global fallback |
| Stale background result | TUI `MentionQuerySource` | Discard by generation |
| Invalid mention range | Notebook manager | Existing validation error |
| ACP composition failure | Notebook manager/TUI submit router | Existing send-time error |

Tracing spans should record root hash, source key, profile, candidate count, matched count, limit, cache hit/miss, build duration, scoring duration, selection duration, and hydration duration. Paths and query text must not be logged verbatim by default.

## 7. Migration and verification strategy

**Z3 Optimize decision:** `phased_tui_facade` — solve `sol_8b3a41b31d6a4546`.

It satisfies behavioral-equivalence gates (12), rollback points (10), and dependency order (9), while accepting more phases than a big-bang migration (4).

### Ordered phases

1. **Characterize behavior**
   - Add golden tests for current TUI typed/empty queries, file traversal, code hydration, worker composition, and URI handling.
   - Preserve notebook completion/root-selection and manager composition tests.

2. **Introduce `spur-mentions` core**
   - Add neutral entry/source/snapshot APIs, filesystem profiles, cache, Nucleo ranker, and bounded top-K.
   - No frontend behavior changes.

3. **Install the TUI façade**
   - Keep the public `spur_tui::mentions::MentionRegistry` surface.
   - Delegate filesystem/cache/ranking to `MentionEngine`.
   - Keep TUI sidecars and policies local.
   - Require golden parity before proceeding.

4. **Move code-mention mechanics**
   - Move discovery, candidate indexes, payload hydration, validation, and expansion behind `code-graph`.
   - Keep protected ranges and ACP block assembly in the TUI submit router.

5. **Integrate the notebook**
   - Add persistent engine state and the thin completion adapter.
   - Use the compatibility profile first.
   - Leave manager validation/composition untouched.

6. **Optimize and benchmark**
   - Enable bounded top-K after equivalence property tests.
   - Measure cold build, warm query, scoring, selection, and hydration at 1k, 10k, and 100k entries.

Each phase is independently revertible. Cross-repository notebook work begins only after the shared crate revision is published and pinned.

## Test matrix, task boundaries, and acceptance criteria

### Shared crate

- Filesystem traversal for both profiles, hidden/ignored files, symlinks, directories, non-ASCII paths, and encoded URIs.
- Root/profile/source-token cache isolation, TTL behavior, invalidation, failed-build retention, and concurrent publication.
- Full-sort versus bounded-top-K equivalence properties.
- Code graph feature-on and feature-off compilation.
- Expansion validation for missing files, changed symbol ranges, caps, and warnings.

### TUI

- Existing picker ordering, section caps, composed worker slots, issue previews, datasource hints, code previews, and protected atoms.
- Background generation/coalescing behavior.
- Submit-router ACP blocks identical before and after extraction.

### Notebook

- Existing recursive completion and root-selection tests.
- Compatibility matching/order and result limit.
- Persistent engine cache hit on repeated queries.
- Existing mention-range validation, attachments, resource links, and ACP composition unchanged.

### Performance acceptance

- Warm-query selection does not perform a full `M log M` sort when `M > K`.
- Output is byte-for-byte equivalent to the reference comparator for the same profile.
- Benchmarks report results rather than setting an invented latency threshold in this spec.

### Plan task boundaries

The future implementation plan should isolate: shared core/API, filesystem/cache, ranking, code feature, TUI façade/adapters, notebook state/adapter, and cross-repository verification. TUI façade depends on shared core; notebook integration depends on a pinned shared revision; performance optimization depends on equivalence tests.

### Release acceptance

- Both repositories build with their prescribed wrapper commands.
- All existing scoped tests pass.
- New parity, cache, and ranking property tests pass.
- Native formal cells in this notebook retain fresh proof hashes.
- No shared-crate dependency on Ratatui, `spur-acp`, `spur-pm`, TUI components, or notebook types.
- The user approves this notebook before implementation planning begins.

## Z3 Optimize evidence

Every decision section used the same discipline: typed-request preflight, hard-feasibility solve, then weighted MaxSMT optimization with `lex` priority and complete termination.

| Section | Selected model | Persisted solve |
|---|---|---|
| Ownership | `split_engine` | `sol_4dcc1f84ce6f4b24` |
| API/data model | `neutral_core_sidecar` | `sol_080f3c2d00ed44f3` |
| Filesystem semantics | `compatibility_profiles` | `sol_d19463fd3c224a70` |
| Cache/concurrency | `injected_arc_engine` | `sol_b1181d8ef6b54c4d` |
| Ranking/performance | `select_nth_then_sort_k` | `sol_d797f83261e24555` |
| Notebook seam | `adapter_exact_rev` | `sol_e72408ccc1a94f60` |
| Migration | `phased_tui_facade` | `sol_8b3a41b31d6a4546` |

These Optimize models select among declared approaches under declared preferences; they do not prove runtime performance. Runtime claims remain gated by tests and benchmarks.

## Review gate

This notebook is a proposed specification until the user reviews it. Approval closes design epic `bd-2n81`; only then may the work transition to a beads-backed implementation plan.